# PCA team embedding — FIFA World Cup 2026 team style metrics

Two components capture most of the spread across style metrics, projecting every team onto a plane. Look for the possession-dominant sides pulling one way and the low-block counter teams the other.

This is a self-contained extract of the PCA section from the full EDA notebook. It only needs `dataset/teams.csv` next to this file.
https://www.kaggle.com/code/swaptr/fifa-world-cup-2026-exploratory-data-analysis/notebook

In [3]:
import os, glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"   # change to "vscode" or "iframe" if you prefer inline rendering
PLOTLY_KW = dict(template="plotly_white")

def _find(fname):
    bases = ["dataset", "."] + sorted(glob.glob("/kaggle/input/*"))
    for b in bases:
        hits = glob.glob(os.path.join(b, "**", fname), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    raise FileNotFoundError(f"{fname} not found. Put teams.csv in a 'dataset' folder next to this notebook.")

teams = pd.read_csv(_find("teams.csv"))

TEAM_ORDER = sorted(teams["team"].dropna().unique())
_pal = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
TEAM_COLOR = {t: _pal[i % len(_pal)] for i, t in enumerate(TEAM_ORDER)}

print(f"teams: {teams.shape}")

teams: (48, 136)


## PCA team embedding

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

PCA_FEATS = ["possession", "shots", "shots_on_target", "goals", "fouls",
             "offsides", "crosses", "interceptions", "tackles_won", "cards_yellow"]
Xp = teams.set_index("team")[PCA_FEATS].astype(float)
Zp = StandardScaler().fit_transform(Xp.values)
pca = PCA(n_components=2, random_state=0)
emb = pca.fit_transform(Zp)
ev = pca.explained_variance_ratio_ * 100

emb85 = pd.DataFrame(emb, columns=["PC1", "PC2"], index=Xp.index).reset_index()
fig = px.scatter(
    emb85, x="PC1", y="PC2", text="team", color="team", color_discrete_map=TEAM_COLOR,
    title=f"PCA of standardized team style metrics (PC1 {ev[0]:.0f}%, PC2 {ev[1]:.0f}% variance)",
    labels={"PC1": f"PC1 ({ev[0]:.0f}%)", "PC2": f"PC2 ({ev[1]:.0f}%)"},
    **PLOTLY_KW,
)
fig.update_traces(textposition="top center", marker=dict(size=11))
fig.update_layout(showlegend=False)
fig.show()

print(f"Cumulative variance PC1+PC2: {ev[:2].sum():.1f}%")

Cumulative variance PC1+PC2: 79.5%
